# AQRS Google Colab Setup
This notebook mounts Google Drive, clones the repo into Drive, installs dependencies, sets `PYTHONPATH`, and verifies parquet access.
Do NOT run full benchmarks from this notebook until you have copied `data/raw` files into Drive.

In [12]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# 2) Clone or unzip the repository into Drive (interactive helper)
import os, subprocess, getpass, re
TARGET = '/content/drive/MyDrive/autonomous-quant-research-system'
if os.path.exists(TARGET) and os.listdir(TARGET):
    print('Target exists and is not empty:', TARGET)
else:
    repo_url = input('Enter GitHub repo HTTPS URL (or leave blank to unzip from Drive): ').strip()
    if repo_url:
        print('Attempting anonymous clone:', repo_url)
        p = subprocess.run(['git', 'clone', repo_url, TARGET], capture_output=True, text=True)
        if p.returncode != 0:
            print('Clone failed:\n', p.stderr)
            if ('could not read Username' in p.stderr) or ('Authentication failed' in p.stderr) or ('fatal: Authentication' in p.stderr):
                token = getpass.getpass('GitHub Personal Access Token (will not be shown): ')
                m = re.match(r'https://github.com/(.+/.+?)(?:\.git)?$', repo_url)
                if m:
                    owner_repo = m.group(1)
                    auth_url = f"https://{token}@github.com/{owner_repo}.git"
                else:
                    auth_url = repo_url.replace('https://', f'https://{token}@')
                print('Attempting authenticated clone...')
                p2 = subprocess.run(['git', 'clone', auth_url, TARGET], capture_output=True, text=True)
                if p2.returncode != 0:
                    print('Authenticated clone failed:\n', p2.stderr)
                else:
                    print('Cloned successfully to', TARGET)
            else:
                print('Clone failed for non-auth reason.')
        else:
            print('Cloned successfully to', TARGET)
    else:
        zip_path = input('Enter Drive zip path (e.g. /content/drive/MyDrive/aqrs.zip): ').strip()
        if not zip_path:
            print('No repo URL or zip path provided; nothing to do.')
        elif not os.path.exists(zip_path):
            print('Zip not found:', zip_path)
        else:
            os.makedirs(TARGET, exist_ok=True)
            p = subprocess.run(['unzip', '-q', zip_path, '-d', TARGET], capture_output=True, text=True)
            if p.returncode != 0:
                print('Unzip failed:\n', p.stderr)
            else:
                print('Unzipped to', TARGET)
print('\nDirectory listing (first 200 entries):')
res = subprocess.run(['ls', '-la', TARGET], capture_output=True, text=True)
print(res.stdout)
if res.stderr: print(res.stderr)

Target exists and is not empty: /content/drive/MyDrive/autonomous-quant-research-system

Directory listing (first 200 entries):
total 25
drwx------ 6 root root 4096 Jun  4 18:45 app
-rw------- 1 root root  246 Jun  4 18:45 ARCHITECTURE.md
-rw------- 1 root root  532 Jun  4 18:45 CHECKLIST.md
drwx------ 4 root root 4096 Jun  4 18:54 data
drwx------ 8 root root 4096 Jun  4 18:45 .git
-rw------- 1 root root  400 Jun  4 18:45 .gitignore
-rw------- 1 root root  720 Jun  4 18:45 README.md
-rw------- 1 root root  374 Jun  4 18:45 requirements.txt
-rw------- 1 root root  803 Jun  4 18:45 ROADMAP.md
drwx------ 2 root root 4096 Jun  4 18:45 tests
drwx------ 2 root root 4096 Jun  4 18:45 tools



In [7]:
# 3) Install requirements
%cd /content/drive/MyDrive/autonomous-quant-research-system
!pip install -r requirements.txt

/content/drive/MyDrive/autonomous-quant-research-system


In [14]:
# 4) Set PYTHONPATH and verify imports
import sys, os
repo = '/content/drive/MyDrive/autonomous-quant-research-system'
if repo not in sys.path:
    sys.path.insert(0, repo)
print('PYTHONPATH set; repo in sys.path:', repo in sys.path)
# quick import smoke-test
try:
    import app
    print('Imported app OK')
except Exception as e:
    print('Import error:', e)

PYTHONPATH set; repo in sys.path: True
Imported app OK


In [ ]:
# 5) Move uploaded 1m.parquet from /content into Drive and verify listing
!mkdir -p /content/drive/MyDrive/autonomous-quant-research-system/data/raw/EURUSD
!if [ -f /content/1m.parquet ]; then mv /content/1m.parquet /content/drive/MyDrive/autonomous-quant-research-system/data/raw/EURUSD/1m.parquet && echo 'Moved 1m.parquet to Drive'; else echo '/content/1m.parquet not found'; fi
!ls -lah /content/drive/MyDrive/autonomous-quant-research-system/data/raw/EURUSD || true

In [9]:
# 5) Verify parquet access (do NOT download new data)
import pandas as pd, os
p = os.path.join(repo, 'data/raw/EURUSD/1m.parquet')
if os.path.exists(p):
    df = pd.read_parquet(p, columns=['timestamp'])
    print('Parquet rows:', len(df), 'max ts:', df['timestamp'].max())
else:
    print('Parquet not found at', p)

Parquet not found at /content/drive/MyDrive/autonomous-quant-research-system/data/raw/EURUSD/1m.parquet
